<a href="https://colab.research.google.com/github/Likith-Reddy25/Summer-Intern/blob/main/ClassicalImpl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [ ]:
!pip install qiskit
!pip install qiskit-machine-learning
!pip install qiskit_algorithms

KeyboardInterrupt: 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import cohen_kappa_score

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

mask = y < 2

X = X[mask]
y = y[mask]

print("Dataset shape:", X.shape)
print("Classes:", np.unique(y))



Dataset shape: (100, 4)
Classes: [0 1]


In [ ]:
models = {

    "Logistic Regression": {

        "model": LogisticRegression(
            max_iter=5000
        ),

        "params": {
            "model__C": [0.01, 0.1, 1, 10, 100]
        }
    },


    "Linear SVM": {

        "model": SVC(
            kernel="linear"
        ),

        "params": {
            "model__C": [0.01, 0.1, 1, 10, 100]
        }
    },


    "Polynomial SVM": {

        "model": SVC(
            kernel="poly"
        ),

        "params": {
            "model__C": [0.01, 0.1, 1, 10, 100]
        }
    },


    "RBF SVM": {

        "model": SVC(
            kernel="rbf"
        ),

        "params": {

            "model__C": [
                0.01,
                0.1,
                1,
                10,
                100
            ],

            "model__gamma": [
                0.001,
                0.01,
                0.1,
                1,
                10
            ]
        }
    },

    "XGBoost": {

    "model": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ),

    "params": {

        "model__n_estimators": [50, 100, 200],

        "model__max_depth": [2, 3, 5],

        "model__learning_rate": [0.01, 0.1, 0.2]
    }
}
}

In [ ]:
results = {}

for name in models:

    results[name] = {

        "train_accuracy": [],
        "test_accuracy": [],

        "train_kappa": [],
        "test_kappa": [],

        "train_f1": [],
        "test_f1": []
    }

In [ ]:
for trial in range(30):

    # New train-test split every trial

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        stratify=y,
        random_state=trial
    )


    for name, config in models.items():

        # Pipeline prevents scaling leakage

        pipeline = Pipeline([

            ("scaler", StandardScaler()),

            ("model", config["model"])

        ])


        # 5-fold cross-validation

        grid = GridSearchCV(
            pipeline,
            config["params"],
            cv=5,
            scoring="accuracy",
            n_jobs=-1
        )


        grid.fit(X_train, y_train)


        best_model = grid.best_estimator_


        # Predictions

        train_pred = best_model.predict(X_train)

        test_pred = best_model.predict(X_test)


        # Store Accuracy

        results[name]["train_accuracy"].append(
            accuracy_score(y_train, train_pred)
        )

        results[name]["test_accuracy"].append(
            accuracy_score(y_test, test_pred)
        )


        # Store Kappa

        results[name]["train_kappa"].append(
            cohen_kappa_score(y_train, train_pred)
        )

        results[name]["test_kappa"].append(
            cohen_kappa_score(y_test, test_pred)
        )


        # Store Macro F1

        results[name]["train_f1"].append(
            f1_score(
                y_train,
                train_pred,
                average="macro"
            )
        )

        results[name]["test_f1"].append(
            f1_score(
                y_test,
                test_pred,
                average="macro"
            )
        )



In [ ]:
print("\nLINEAR IRIS — AVERAGE OF 30 TRIALS\n")


for name, r in results.items():

    print("=" * 55)
    print(name)
    print("=" * 55)


    print(
        f"Train Accuracy : "
        f"{np.mean(r['train_accuracy']):.4f} ± "
        f"{np.std(r['train_accuracy'], ddof=1):.4f}"
    )


    print(
        f"Test Accuracy  : "
        f"{np.mean(r['test_accuracy']):.4f} ± "
        f"{np.std(r['test_accuracy'], ddof=1):.4f}"
    )


    print(
        f"Train Kappa    : "
        f"{np.mean(r['train_kappa']):.4f} ± "
        f"{np.std(r['train_kappa'], ddof=1):.4f}"
    )


    print(
        f"Test Kappa     : "
        f"{np.mean(r['test_kappa']):.4f} ± "
        f"{np.std(r['test_kappa'], ddof=1):.4f}"
    )


    print(
        f"Train F1       : "
        f"{np.mean(r['train_f1']):.4f} ± "
        f"{np.std(r['train_f1'], ddof=1):.4f}"
    )


    print(
        f"Test F1        : "
        f"{np.mean(r['test_f1']):.4f} ± "
        f"{np.std(r['test_f1'], ddof=1):.4f}"
    )

    print()

In [ ]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

In [ ]:
qke_results = {
    "train_accuracy": [],
    "test_accuracy": [],
    "train_kappa": [],
    "test_kappa": [],
    "train_f1": [],
    "test_f1": []
}


C_range = np.logspace(-2, 2, 5)

scaling_factor_range = [
    0.001,
    0.01,
    0.1,
    0.5,
    1.0
]


random_seed = 12345

In [ ]:
for rep in range(30):

    # ========================================================
    # 1. NEW 70/30 SPLIT
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_seed,
        stratify=y
    )

    random_seed += 1


    # ========================================================
    # 2. QUANTUM INPUT SCALING
    # [0, 2π]
    # ========================================================

    scaler = MinMaxScaler(
        feature_range=(0, 2 * np.pi)
    )

    X_train_q = scaler.fit_transform(X_train)
    X_test_q = scaler.transform(X_test)


    # ========================================================
    # 3. ZZ FEATURE MAP
    # ========================================================

    feature_map = ZZFeatureMap(
        feature_dimension=4,
        reps=2,
        insert_barriers=True
    )


    # ========================================================
    # 4. CROSS VALIDATION
    # ========================================================

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=False
    )


    best_score = None
    best_C = None
    best_scaling_factor = None


    # ========================================================
    # 5. BANDWIDTH SEARCH
    # ========================================================

    for scaling_factor in scaling_factor_range:

        # This is how the paper applies quantum bandwidth

        X_train_scaled = X_train_q * scaling_factor


        # Quantum kernel

        quantum_kernel = FidelityStatevectorKernel(
            feature_map=feature_map
        )


        # Training kernel matrix

        K_train = quantum_kernel.evaluate(
            x_vec=X_train_scaled
        )


        # ====================================================
        # C SEARCH USING 5-FOLD CV
        # ====================================================

        param_grid = {
            "C": C_range
        }


        grid = GridSearchCV(
            SVC(kernel="precomputed"),
            param_grid=param_grid,
            cv=cv,
            refit=True
        )


        grid.fit(
            K_train,
            y_train
        )


        current_score = grid.best_score_


        # ====================================================
        # SAVE BEST λ AND C
        # ====================================================

        if best_score is None or current_score > best_score:

            best_score = current_score

            best_scaling_factor = scaling_factor

            best_C = grid.best_params_["C"]


    # ========================================================
    # 6. USE BEST BANDWIDTH
    # ========================================================

    X_train_best = (
        X_train_q * best_scaling_factor
    )

    X_test_best = (
        X_test_q * best_scaling_factor
    )


    quantum_kernel = FidelityStatevectorKernel(
        feature_map=feature_map
    )


    K_train_best = quantum_kernel.evaluate(
        x_vec=X_train_best
    )


    K_test_best = quantum_kernel.evaluate(
        x_vec=X_test_best,
        y_vec=X_train_best
    )


    # ========================================================
    # 7. FINAL QSVM
    # ========================================================

    qsvc = SVC(
        kernel="precomputed",
        C=best_C
    )


    qsvc.fit(
        K_train_best,
        y_train
    )


    train_pred = qsvc.predict(
        K_train_best
    )

    test_pred = qsvc.predict(
        K_test_best
    )


    # ========================================================
    # 8. STORE RESULTS
    # ========================================================

    qke_results["train_accuracy"].append(
        accuracy_score(y_train, train_pred)
    )

    qke_results["test_accuracy"].append(
        accuracy_score(y_test, test_pred)
    )


    qke_results["train_kappa"].append(
        cohen_kappa_score(y_train, train_pred)
    )

    qke_results["test_kappa"].append(
        cohen_kappa_score(y_test, test_pred)
    )


    qke_results["train_f1"].append(
        f1_score(
            y_train,
            train_pred,
            average="macro"
        )
    )

    qke_results["test_f1"].append(
        f1_score(
            y_test,
            test_pred,
            average="macro"
        )
    )

/tmp/ipykernel_456/2275761919.py:35: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(
/tmp/ipykernel_456/2275761919.py:35: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(
/tmp/ipykernel_456/2275761919.py:35: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a repla

In [ ]:
print("\nZZFeatureMap + QKE — 30 Trials\n")


for metric, values in qke_results.items():

    print(
        f"{metric:20s}: "
        f"{np.mean(values):.4f} ± "
        f"{np.std(values, ddof=1):.4f}"
    )


ZZFeatureMap + QKE — 30 Trials

train_accuracy      : 1.0000 ± 0.0000
test_accuracy       : 1.0000 ± 0.0000
train_kappa         : 1.0000 ± 0.0000
test_kappa          : 1.0000 ± 0.0000
train_f1            : 1.0000 ± 0.0000
test_f1             : 1.0000 ± 0.0000


In [ ]:
# ============================================================
# QUANTUM-SPECIFIC IMPORTS FOR QKT
# ============================================================

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map

from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms.optimizers import SPSA

from qiskit_machine_learning.kernels import (
    FidelityStatevectorKernel,
    TrainableFidelityStatevectorKernel
)

from qiskit_machine_learning.kernels.algorithms import (
    QuantumKernelTrainer
)

from qiskit_machine_learning.algorithms import QSVC

from qiskit_machine_learning.utils.loss_functions import (
    SVCLoss
)

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold


# ============================================================
# CALLBACK
# Stores every SPSA point and loss
# ============================================================

class QKTCallback:

    def __init__(self):
        self.params = []
        self.losses = []

    def callback(
        self,
        nfev,
        parameters,
        value,
        stepsize,
        accepted
    ):

        self.params.append(
            np.array(parameters).copy()
        )

        self.losses.append(value)


# ============================================================
# SHARED TRAINABLE FEATURE MAP
#
# Same 3 parameters are applied to every qubit
# ============================================================

def create_shared_feature_map(
    base_feature_map,
    training_params
):

    rotation_layer = QuantumCircuit(
        base_feature_map.num_qubits
    )


    for qubit in range(
        base_feature_map.num_qubits
    ):

        rotation_layer.u(
            training_params[0],
            training_params[1],
            training_params[2],
            qubit
        )


    # V(theta) followed by ZZFeatureMap

    trainable_map = rotation_layer.compose(
        base_feature_map
    )


    return trainable_map


# ============================================================
# RESULT STORAGE
# ============================================================

shared_results = {

    "train_accuracy": [],
    "test_accuracy": [],

    "train_kappa": [],
    "test_kappa": [],

    "train_f1": [],
    "test_f1": []
}


# ============================================================
# PAPER HYPERPARAMETERS
# ============================================================

C_range = np.logspace(-2, 2, 5)


scaling_factor_range = [

    0.001,
    0.01,
    0.1,
    0.5,
    1.0

]


random_seed = 12345

algorithm_globals.random_seed = random_seed


# ============================================================
# 30 REPETITIONS
# ============================================================

for rep in range(5):


    # ========================================================
    # 1. NEW TRAIN-TEST SPLIT
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(

        X,
        y,

        test_size=0.30,

        random_state=random_seed,

        stratify=y

    )


    random_seed += 1


    # ========================================================
    # 2. QUANTUM INPUT SCALING
    # ========================================================

    scaler = MinMaxScaler(

        feature_range=(0, 2 * np.pi)

    )


    X_train_q = scaler.fit_transform(
        X_train
    )

    X_test_q = scaler.transform(
        X_test
    )


    # ========================================================
    # 3. BASE ZZ FEATURE MAP
    # ========================================================

    zz_map = zz_feature_map(

        feature_dimension=4,

        reps=2,

        insert_barriers=True

    )


    # ========================================================
    # 4. FIND BEST C AND BANDWIDTH
    #
    # Same QKE search used before QKT in paper
    # ========================================================

    cv = StratifiedKFold(

        n_splits=5,

        shuffle=False

    )


    best_score = None

    best_C = None

    best_scaling_factor = None


    for scaling_factor in scaling_factor_range:


        X_train_scaled = (

            X_train_q * scaling_factor

        )


        qke_kernel = FidelityStatevectorKernel(

            feature_map=zz_map

        )


        K_train = qke_kernel.evaluate(

            x_vec=X_train_scaled

        )


        grid = GridSearchCV(

            SVC(kernel="precomputed"),

            param_grid={
                "C": C_range
            },

            cv=cv,

            refit=True

        )


        grid.fit(

            K_train,

            y_train

        )


        current_score = grid.best_score_


        if (
            best_score is None
            or current_score > best_score
        ):

            best_score = current_score

            best_scaling_factor = (
                scaling_factor
            )

            best_C = (
                grid.best_params_["C"]
            )


    # ========================================================
    # 5. APPLY BEST BANDWIDTH
    # ========================================================

    X_train_best = (

        X_train_q
        * best_scaling_factor

    )


    X_test_best = (

        X_test_q
        * best_scaling_factor

    )


    # ========================================================
    # 6. CREATE SHARED PARAMETERS
    #
    # Exactly 3 trainable parameters
    # ========================================================

    theta = ParameterVector(

        "θ",

        3

    )


    initial_point = np.zeros(3)


    # ========================================================
    # 7. CREATE TRAINABLE FEATURE MAP
    # ========================================================

    trainable_map = create_shared_feature_map(

        zz_map,

        theta

    )


    # ========================================================
    # 8. TRAINABLE QUANTUM KERNEL
    # ========================================================

    trainable_kernel = (

        TrainableFidelityStatevectorKernel(

            feature_map=trainable_map,

            training_parameters=theta

        )

    )


    # ========================================================
    # 9. SPSA
    #
    # Paper:
    # maxiter = 400
    # blocking = True
    # second_order = True
    # ========================================================

    callback = QKTCallback()


    optimizer = SPSA(

        maxiter=400,

        callback=callback.callback,

        blocking=True,

        second_order=True

    )


    # ========================================================
    # 10. WEIGHTED SVC LOSS
    # ========================================================

    loss = SVCLoss(

        C=best_C

    )


    # ========================================================
    # 11. QUANTUM KERNEL TRAINER
    # ========================================================

    qkt = QuantumKernelTrainer(

        quantum_kernel=trainable_kernel,

        loss=loss,

        optimizer=optimizer,

        initial_point=initial_point

    )


    # Reproducible SPSA randomness

    algorithm_globals.random_seed = (
        random_seed
    )


    # ========================================================
    # 12. TRAIN QKT
    # ========================================================

    qkt.fit(

        X_train_best,

        y_train

    )


    # ========================================================
    # 13. FIND MINIMUM-LOSS POINT
    #
    # Paper uses best point visited by SPSA,
    # not necessarily final point
    # ========================================================

    best_index = np.argmin(

        callback.losses

    )


    best_theta = (

        callback.params[best_index]

    )


    # Assign best θ values

    trainable_kernel.assign_training_parameters(

        best_theta

    )


    # ========================================================
    # 14. FINAL QSVC
    # ========================================================

    qsvc = QSVC(

        quantum_kernel=trainable_kernel,

        C=best_C

    )


    qsvc.fit(

        X_train_best,

        y_train

    )


    # ========================================================
    # 15. PREDICTIONS
    # ========================================================

    train_pred = qsvc.predict(

        X_train_best

    )


    test_pred = qsvc.predict(

        X_test_best

    )


    # ========================================================
    # 16. STORE METRICS
    # ========================================================

    shared_results[
        "train_accuracy"
    ].append(

        accuracy_score(
            y_train,
            train_pred
        )

    )


    shared_results[
        "test_accuracy"
    ].append(

        accuracy_score(
            y_test,
            test_pred
        )

    )


    shared_results[
        "train_kappa"
    ].append(

        cohen_kappa_score(
            y_train,
            train_pred
        )

    )


    shared_results[
        "test_kappa"
    ].append(

        cohen_kappa_score(
            y_test,
            test_pred
        )

    )


    shared_results[
        "train_f1"
    ].append(

        f1_score(
            y_train,
            train_pred,
            average="macro"
        )

    )


    shared_results[
        "test_f1"
    ].append(

        f1_score(
            y_test,
            test_pred,
            average="macro"
        )

    )


    print(
        f"Completed repetition "
        f"{rep + 1}/30"
    )


# ============================================================
# FINAL AVERAGE OF 30 TRIALS
# ============================================================

print(
    "\nZZFeatureMap + QKT Shared (3)"
)

print(
    "Average of 30 trials\n"
)


for metric, values in shared_results.items():

    print(

        f"{metric:20s}: "

        f"{np.mean(values):.4f} ± "

        f"{np.std(values, ddof=1):.4f}"

    )

KeyboardInterrupt: 

In [ ]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map

from qiskit_machine_learning.kernels import (
    TrainableFidelityStatevectorKernel
)

from qiskit_machine_learning.kernels.algorithms import (
    QuantumKernelTrainer
)

from qiskit_machine_learning.utils.loss_functions import SVCLoss

from qiskit_algorithms.optimizers import SPSA

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score
)


# ============================================================
# RESULTS
# ============================================================

qkt_dedicated_results = {
    "train_accuracy": [],
    "test_accuracy": [],
    "train_kappa": [],
    "test_kappa": [],
    "train_f1": [],
    "test_f1": []
}


# ============================================================
# HYPERPARAMETER RANGES
# ============================================================

C_range = np.logspace(-2, 2, 5)

scaling_factor_range = [
    0.001,
    0.01,
    0.1,
    0.5,
    1.0
]


random_seed = 12345


# ============================================================
# 30 REPETITIONS
# ============================================================

for rep in range(5):

    print(f"\nRepetition {rep + 1}/30")


    # ========================================================
    # 1. NEW 70/30 SPLIT
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_seed,
        stratify=y
    )

    random_seed += 1


    # ========================================================
    # 2. QUANTUM INPUT SCALING
    # [0, 2π]
    # ========================================================

    scaler = MinMaxScaler(
        feature_range=(0, 2 * np.pi)
    )

    X_train_q = scaler.fit_transform(X_train)
    X_test_q = scaler.transform(X_test)


    # ========================================================
    # 3. CROSS VALIDATION
    # ========================================================

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=False
    )


    best_score = None
    best_C = None
    best_scaling_factor = None
    best_trained_kernel = None


    # ========================================================
    # 4. BANDWIDTH SEARCH
    # ========================================================

    for scaling_factor in scaling_factor_range:

        print(
            f"  Testing scaling factor = {scaling_factor}"
        )


        X_train_scaled = (
            X_train_q * scaling_factor
        )


        # ====================================================
        # 5. DEDICATED TRAINABLE CIRCUIT
        #
        # 4 qubits × 3 parameters = 12 parameters
        # ====================================================

        num_qubits = 4

        theta = ParameterVector(
            "θ",
            length=3 * num_qubits
        )


        trainable_feature_map = QuantumCircuit(
            num_qubits
        )


        # ----------------------------------------------------
        # V_dedicated(theta)
        #
        # Each qubit gets its own 3 parameters
        # ----------------------------------------------------

        for q in range(num_qubits):

            trainable_feature_map.u(
                theta[3 * q],
                theta[3 * q + 1],
                theta[3 * q + 2],
                q
            )


        # ----------------------------------------------------
        # ZZFeatureMap Ux
        # ----------------------------------------------------

        zz_map = zz_feature_map(
            feature_dimension=num_qubits,
            reps=2,
            insert_barriers=True
        )


        # V(theta) followed by U_ZZ(x)

        trainable_feature_map.compose(
            zz_map,
            inplace=True
        )


        # ====================================================
        # 6. TRAINABLE QUANTUM KERNEL
        # ====================================================

        trainable_kernel = (
            TrainableFidelityStatevectorKernel(
                feature_map=trainable_feature_map,
                training_parameters=list(theta)
            )
        )


        # ====================================================
        # 7. SPSA OPTIMIZER
        #
        # Paper: maximum 400 iterations
        # ====================================================

        optimizer = SPSA(
            maxiter=400,
            second_order=True
        )


        # ====================================================
        # 8. QUANTUM KERNEL TRAINER
        #
        # SVCLoss = weighted SVM-based alignment objective
        # ====================================================

        qkt = QuantumKernelTrainer(
            quantum_kernel=trainable_kernel,
            loss=SVCLoss(),
            optimizer=optimizer,
            initial_point=np.zeros(
                len(theta)
            )
        )


        # ====================================================
        # 9. TRAIN QKT
        # ====================================================

        qkt_result = qkt.fit(
            X_train_scaled,
            y_train
        )


        trained_kernel = (
            qkt_result.quantum_kernel
        )


        # ====================================================
        # 10. TRAINING KERNEL MATRIX
        # ====================================================

        K_train = trained_kernel.evaluate(
            x_vec=X_train_scaled
        )


        # ====================================================
        # 11. C SEARCH USING 5-FOLD CV
        # ====================================================

        param_grid = {
            "C": C_range
        }


        grid = GridSearchCV(
            SVC(kernel="precomputed"),
            param_grid=param_grid,
            cv=cv,
            refit=True
        )


        grid.fit(
            K_train,
            y_train
        )


        current_score = grid.best_score_


        # ====================================================
        # 12. SAVE BEST λ, C AND TRAINED KERNEL
        # ====================================================

        if (
            best_score is None
            or current_score > best_score
        ):

            best_score = current_score

            best_scaling_factor = (
                scaling_factor
            )

            best_C = (
                grid.best_params_["C"]
            )

            best_trained_kernel = (
                trained_kernel
            )


    # ========================================================
    # 13. USE BEST BANDWIDTH
    # ========================================================

    X_train_best = (
        X_train_q
        * best_scaling_factor
    )

    X_test_best = (
        X_test_q
        * best_scaling_factor
    )


    # ========================================================
    # 14. FINAL KERNEL MATRICES
    # ========================================================

    K_train_best = (
        best_trained_kernel.evaluate(
            x_vec=X_train_best
        )
    )


    K_test_best = (
        best_trained_kernel.evaluate(
            x_vec=X_test_best,
            y_vec=X_train_best
        )
    )


    # ========================================================
    # 15. FINAL QSVM
    # ========================================================

    qsvc = SVC(
        kernel="precomputed",
        C=best_C
    )


    qsvc.fit(
        K_train_best,
        y_train
    )


    train_pred = qsvc.predict(
        K_train_best
    )

    test_pred = qsvc.predict(
        K_test_best
    )


    # ========================================================
    # 16. STORE RESULTS
    # ========================================================

    qkt_dedicated_results[
        "train_accuracy"
    ].append(
        accuracy_score(
            y_train,
            train_pred
        )
    )


    qkt_dedicated_results[
        "test_accuracy"
    ].append(
        accuracy_score(
            y_test,
            test_pred
        )
    )


    qkt_dedicated_results[
        "train_kappa"
    ].append(
        cohen_kappa_score(
            y_train,
            train_pred
        )
    )


    qkt_dedicated_results[
        "test_kappa"
    ].append(
        cohen_kappa_score(
            y_test,
            test_pred
        )
    )


    qkt_dedicated_results[
        "train_f1"
    ].append(
        f1_score(
            y_train,
            train_pred,
            average="macro"
        )
    )


    qkt_dedicated_results[
        "test_f1"
    ].append(
        f1_score(
            y_test,
            test_pred,
            average="macro"
        )
    )


    print(
        f"  Best λ = {best_scaling_factor}, "
        f"Best C = {best_C}, "
        f"Test Accuracy = "
        f"{qkt_dedicated_results['test_accuracy'][-1]:.4f}"
    )


# ============================================================
# FINAL RESULTS
# ============================================================

print(
    "\nZZFeatureMap + QKT Dedicated (12) — 30 Trials\n"
)


for metric, values in qkt_dedicated_results.items():

    print(
        f"{metric:20s}: "
        f"{np.mean(values):.4f} ± "
        f"{np.std(values, ddof=1):.4f}"
    )


Repetition 1/30
  Testing scaling factor = 0.001
  Testing scaling factor = 0.01
  Testing scaling factor = 0.1
  Testing scaling factor = 0.5
  Testing scaling factor = 1.0
  Best λ = 0.001, Best C = 100.0, Test Accuracy = 1.0000

Repetition 2/30
  Testing scaling factor = 0.001
  Testing scaling factor = 0.01
  Testing scaling factor = 0.1
  Testing scaling factor = 0.5
  Testing scaling factor = 1.0
  Best λ = 0.001, Best C = 100.0, Test Accuracy = 1.0000

Repetition 3/30
  Testing scaling factor = 0.001
  Testing scaling factor = 0.01
  Testing scaling factor = 0.1
  Testing scaling factor = 0.5
  Testing scaling factor = 1.0
  Best λ = 0.001, Best C = 100.0, Test Accuracy = 1.0000

Repetition 4/30
  Testing scaling factor = 0.001
  Testing scaling factor = 0.01
  Testing scaling factor = 0.1
  Testing scaling factor = 0.5
